<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula10f%20-%20lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* “What’s the topic of this text?” (text classification)
* “Does this text contain abuse?” (content filtering)
* “Does this text sound positive or negative?” **(sentiment analysis)**
* “What should be the next word in this incomplete sentence?” (language modeling)
* “How would you say this in German?” (translation)
* “How would you summarize this article in one paragraph?” (summarization)

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  3477k      0  0:00:23  0:00:23 --:--:-- 5039k


In [2]:
!rm -r aclImdb/train/unsup

In [3]:
!cat aclImdb/train/pos/4077_10.txt

I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drink and food in reach so you don't have to stop the film.<br /><br />Enjoy

In [4]:
import os, pathlib, shutil, random

base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"
for category in ("neg", "pos"):
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)

In [5]:
from tensorflow import keras
batch_size = 32

print("train_ds")
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size
)
print("val_ds")
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size
)
print("test_ds")
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size
)

train_ds
Found 20000 files belonging to 2 classes.
val_ds
Found 5000 files belonging to 2 classes.
test_ds
Found 25000 files belonging to 2 classes.


In [6]:
for inputs, targets in train_ds:
  print("inputs.shape:", inputs.shape)
  print("inputs.dtype:", inputs.dtype)
  print("targets.shape:", targets.shape)
  print("targets.dtype:", targets.dtype)
  print("inputs[0]:", inputs[0])
  print("targets[0]:", targets[0])
  break

inputs.shape: (32,)
inputs.dtype: <dtype: 'string'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor(b'Normally, I have much better things to do with my time than write reviews but I was so disappointed with this movie that I spent an hour registering with IMDb just to get it off my chest.<br /><br />You would think a movie with names like Morgan Freeman or Kevin Spacey would be a bankable bet... well, this movie was just terrible. It is nigh on impossible to "suspend disbelief"; I tried, really, I wanted to enjoy it but Justin Timberlake just wouldn\'t let me. <br /><br />Timberlake should stick to music, what a dreadful performance - NO presence as an actor,NO character. Can\'t blame everything on Justin: The movie also boast a dreadful plot & badly timed editing; its definitely an "F".<br /><br />After seeing this, I have to wonder what really motivates actors. I mean, surely Morgan actually read the script before taking the part. Did he not see how poor it 

In [26]:
from tensorflow.keras.layers import TextVectorization

max_length = 600
max_tokens = 20000

text_vectorization = TextVectorization(max_tokens=max_tokens,
                                       output_mode="int",
                                       output_sequence_length=max_length)

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

binary_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [27]:
for inputs, targets in binary_train_ds:
  print("inputs.shape:", inputs.shape)
  print("inputs.dtype:", inputs.dtype)
  print("targets.shape:", targets.shape)
  print("targets.dtype:", targets.dtype)
  print("inputs[0]:", inputs[0])
  print("targets[0]:", targets[0])
  break

inputs.shape: (32, 600)
inputs.dtype: <dtype: 'int64'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor(
[   11    19     7    53    85     2   113     7   476    17   311   937
     5  9271  7910    15     2     1  3430    37  5968    46     4 12208
    23  5127     8     2   436    27  1019    23   314    17     2     1
     4    91    27   260   919    27    77   289  1093    21    23   314
     4     2     1  5908     2  5127    27 12208     2    88   309   300
    24    77   287   253    47    24    63  1268    39     3     1  4363
    12    44  3064    16   474   313   985     4     7   124   814     8
     2  1810    18     2  1798   137    24   416  6082   148    11    19
    44     2   837   207    12     9     7    53   592   196     2  1351
   515    24   524    41     1   359   515    18    16     2   146     1
    11     7     3     1     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0 

In [29]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=20000, hidden_dim=32):
  inputs = keras.Input(shape=(None,), dtype="int64")
  embedded = layers.Embedding(input_dim=max_tokens,
                              output_dim=64,
                              mask_zero=True)(inputs)
  x = layers.Bidirectional(layers.LSTM(hidden_dim))(embedded)
  x = layers.Dropout(0.5)(x)
  outputs = layers.Dense(1, activation="sigmoid")(x)
  model = keras.Model(inputs, outputs)
  model.compile(optimizer="rmsprop",
                loss="binary_crossentropy",
                metrics=["accuracy"])
  return model

model = get_model()
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │  1,280,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_6[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 64)        │     24,832 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         65 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,304,897 (4.98 MB)

 Trainable params: 1,304,897 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
callbacks = [
    keras.callbacks.ModelCheckpoint("lstm.keras",
                                    save_best_only=True)
]
model.fit(binary_train_ds,
          validation_data=binary_val_ds,
          epochs=10,
          callbacks=callbacks)
model = keras.models.load_model("lstm.keras")
test_loss, test_acc = model.evaluate(binary_test_ds)
print(f"Test Loss: {test_loss:.3f}")
print(f"Test Accuracy: {test_acc:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 49ms/step - accuracy: 0.7653 - loss: 0.4676 - val_accuracy: 0.8626 - val_loss: 0.3280
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.8831 - loss: 0.2899 - val_accuracy: 0.8684 - val_loss: 0.3136
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9103 - loss: 0.2315 - val_accuracy: 0.8854 - val_loss: 0.2897
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.9254 - loss: 0.1933 - val_accuracy: 0.8626 - val_loss: 0.3175
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9419 - loss: 0.1581 - val_accuracy: 0.8722 - val_loss: 0.3283
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9549 - loss: 0.1313 - val_accuracy: 0.8326 - val_loss: 0.4858
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.9632 - loss: 0.1055 - val_accuracy: 0.8690 - val_loss: 0.3664
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9718 - loss: 0.0829 - 